# Introduction
This notebook will use PySpark's MLlib to build and compare supervised learning models on a gene
expression dataset for leukemia classification. First, we read in the data, split it into training and test
sets, and define an evaluation metric. We then fit three model classes (Random Forest, Gradient-Boosted
Tree, and Multilayer Perceptron) using pipelines and cross-validation to select the best
hyperparameters for each. Finally, we evaluate each best model on the held-out test set and determine
an overall winner.

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, PCA
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [3]:
spark = (SparkSession.builder
    .appName("GeneExpression")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 21:23:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv("leukemia_gene_expression.csv")

In [5]:
# Identify columns
label_col = df.columns[-1]
feature_cols = [c for c in df.columns if c != label_col]

# Train/Test split
train, test = df.randomSplit([0.8, 0.2], seed=42)

# Pipeline: 4 transformations + 1 estimator
label_indexer = StringIndexer(inputCol=label_col, outputCol="label")
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled", withStd=True, withMean=True)
pca = PCA(inputCol="features_scaled", outputCol="features")
rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=42)

pipeline = Pipeline(stages=[label_indexer, assembler, scaler, pca, rf])

# Cross-validation
param_grid = (ParamGridBuilder()
    .addGrid(pca.k, [10, 30, 50])
    .addGrid(rf.numTrees, [50, 100, 200])
    .addGrid(rf.maxDepth, [5, 10, 15])
    .build())

evaluator = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")

cv = CrossValidator(estimator=pipeline, estimatorParamMaps=param_grid,
                    evaluator=evaluator, numFolds=5, seed=42, parallelism=16)

cv_model = cv.fit(train)

# Evaluate
rf_predictions = cv_model.transform(test)
print(f"Random Forest — Test F1: {evaluator.evaluate(rf_predictions):.4f}")

# Best hyperparameters
best = cv_model.bestModel
print(f"Best PCA k:    {best.stages[3].getK()}")
print(f"Best numTrees: {best.stages[4].getNumTrees}")
print(f"Best maxDepth: {best.stages[4].getOrDefault('maxDepth')}")

Random Forest — Test F1: 0.3833
Best PCA k:    10
Best numTrees: 200
Best maxDepth: 15
